## Setup 

In [ ]:
SNT_ROOT_PATH <- "~/workspace"

In [ ]:
source(file.path(SNT_ROOT_PATH, "pipelines/snt_dhis2_formatting/utils/snt_dhis2_formatting.r"))
setup_var <- get_setup_variables(SNT_ROOT_PATH = SNT_ROOT_PATH, packages=c("lubridate", "zoo", "glue", "arrow", "dplyr", "tidyr", "stringr", "stringi", "jsonlite", "reticulate"))
config_json <- load_snt_config(file.path(setup_var$CONFIG_PATH, "SNT_config.json"))

# Save this country code in a variable
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
ADMIN_1 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_1)
ADMIN_2 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_2)

### Load DHIS2 analytics data  

-Load DHIS2 anlytics from latest dataset version 

In [ ]:
# DHIS2 Dataset extract identifier
dataset_name <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_EXTRACTS
dhis2_data <- load_dataset_file(dataset_name, paste0(COUNTRY_CODE, "_dhis2_raw_analytics.parquet"))

## SNT Indicators computation

### Select DHIS2 metadata  

In [ ]:
# log
log_msg("Computing SNT indicators.")

# Select only metadata (reduce the size of the dataframe)
administrative_cols <- colnames(dhis2_data)[grepl("LEVEL_", colnames(dhis2_data))]
dhis2_metadata <- dhis2_data[ , c("OU", "DX_NAME", administrative_cols)] # Metadata
dhis2_metadata <- distinct(dhis2_metadata)
print(glue("Data element names: \n {paste(unique(dhis2_metadata$DX_NAME), collapse=',\n ')}"))

# Clean strings for admin 1 and admin 2 (format_names() in snt_utils.r)
dhis2_metadata[[ADMIN_1]] <- format_names(dhis2_metadata[[ADMIN_1]]) 
dhis2_metadata[[ADMIN_2]] <- format_names(dhis2_metadata[[ADMIN_2]])

In [ ]:
# Max admin columns available (matchin ou)
name_cols <- grep("LEVEL_\\d+_NAME", administrative_cols, value = TRUE)
max_level <- max(as.numeric(gsub("LEVEL_(\\d+)_NAME", "\\1", name_cols)))
max_admin_col_name <- paste0("LEVEL_", max_level, "_NAME")
print(glue("Max admin level available: '{max_admin_col_name}'"))

### Pivot DHIS2 value table

In [ ]:
# dhis2 Values table
dhis2_values <- dhis2_data[ , c("DX", "CO", "OU", "PE", "VALUE")]

# make sure we have numeric data in "values" column
dhis2_values$VALUE <- as.numeric(dhis2_values$VALUE)

# pivot table on DX and CO columns (available combinations to columns)
routine_data <- pivot_wider(dhis2_values,
                            id_cols = all_of(c("OU", "PE")),
                            names_from = c("DX", "CO"),
                            values_from = 'VALUE')

print(paste("Routine data pivot : ", paste0(dim(routine_data), collapse=", ")))

### Build indicator definitions

**Separate valid from empty indicators definitions from the configuration list**

In [ ]:
dhis_indicator_definitions <- config_json$DHIS2_DATA$DHIS2_INDICATOR_DEFINITIONS
for (n in names(dhis_indicator_definitions)) {
  cat(n, ":", paste(dhis_indicator_definitions[[n]], collapse = ", "), "\n")
}

In [ ]:
# Select the list of valid indicators from the definitions: 
# -valid
# -empty (no data elements defined)
indicators_selection_result <- indicators_selection(dhis_indicator_definitions)

print(glue("\n{paste(c('Complete indicator definitions:', names(indicators_selection_result$valid_indicators)), collapse='\n -')}"))
print(glue("\n{paste(c('Empty indicator definitions:', indicators_selection_result$empty_indicators), collapse='\n -')}"))

**Start indicators loop**

In [ ]:
# build SNT indicators 
routine_data_ind <- build_indicators(
    data=routine_data, 
    valid_indicators=indicators_selection_result$valid_indicators,
    empty_indicators=indicators_selection_result$empty_indicators
)

print(dim(routine_data_ind))
head(routine_data_ind, 3)

## Format SNT routine data

### SNT format 

In [ ]:
# Merge back the metadata and apply final format to the routine data.
routine_data_formatted <- merge_and_format_routine_data(
    data=routine_data_ind,    
    metadata=dhis2_metadata,
    indicator_definitions=dhis_indicator_definitions
)

print(dim(routine_data_formatted))
head(routine_data_formatted, 3)

### Output formatted routine data

In [ ]:
FORMATTED_DATA_PATH <- file.path(setup_var$DATA_PATH, "dhis2", "extracts_formatted")

# write files
write_parquet(routine_data_formatted, file.path(FORMATTED_DATA_PATH, paste0(COUNTRY_CODE, "_routine.parquet")))
write.csv(routine_data_formatted, file.path(FORMATTED_DATA_PATH, paste0(COUNTRY_CODE, "_routine.csv")), row.names = FALSE)
log_msg(glue("Rountine data saved under: {file.path(FORMATTED_DATA_PATH, paste0(COUNTRY_CODE, '_routine.parquet'))}"))

### Data Summary 

In [ ]:
# Data summary
print(summary(routine_data_formatted))